# Playing pokelike from Python

A game runs in a headless browser: no window, no internet, no pixels are ever
read. `state` is a JavaScript object in memory and the buttons are DOM objects,
both of which exist perfectly well without anything being drawn.

A notebook is the awkward case for the usual `with` block, because `with` cannot
span cells — and starting a run in one cell, taking a move in the next and reading
the state in the one after is the whole point of using a notebook. So `open_game()`
hands back a game that stays alive until you close it.

Run this once first, if you have not already: `uv run pokelike setup`


## Open a game

This starts a local web server for the copy of the game on disk, then a browser
pointed at it. The port is asked for from the OS, so two notebooks never collide.


In [ ]:
from pokelike import open_game

game = open_game()
game


## Start a run

The seed pins everything: the map, the encounters, the items on offer. Same seed
and same moves give the same run, score included.

Picking the trainer and the starter are NOT done for you — they are the first two
decisions of the run.


In [ ]:
obs = game.reset(seed=42)
obs['screen'], obs['steps'], obs['done']


## What a bot sees

One dict. `actions` is the list you choose from, and its indices are what `step`
takes. `uv run pokelike schema` prints the full reference, generated from a live
observation.


In [ ]:
obs['actions']


In [ ]:
sorted(obs.keys())


## The same thing, readable

`render` rebuilds all of it from the state. Nothing here is read off a picture:
the map below is drawn from the nodes and edges.


In [ ]:
from pokelike.core import render

print(render.screen(obs))


## Take a move

`step` takes an index into `obs['actions']` and returns the new state. Between one
decision and the next the engine does plenty on its own — plays out the battle,
shows level-ups, banners — and none of that is a choice, so `step` only hands
control back when there is really something to decide.


In [ ]:
obs = game.step(0)
print(obs['screen'], '|', len(obs['actions']), 'actions')


Run this cell again and again to walk forward one decision at a time.


In [ ]:
obs = game.step(0)
print(render.screen(obs))


## The map is a graph

Choosing a node closes every other one on its layer **forever**, so where a node
leads matters as much as what it is.


In [ ]:
print(render.graph_view(obs['map'], colour=False))


## Team order is a decision, and it is free

Slot 0 leads the next battle. Reordering does not consume the turn, which is why
it is its own verb rather than one of `actions`: a full team would otherwise add
fifteen swap pairs beside the real moves at every single map node.


In [ ]:
if obs.get('can_reorder'):
    obs = game.reorder(0, 1)
    print(render.team_view(obs['team']))
else:
    print('not enough Pokemon yet')


## The score

The game's own formula, not one we invented. Compare with `points_no_time`: the
time bonus depends on the clock, which is frozen for reproducibility, so it sits
pinned near 1000 and would drown out everything else.


In [ ]:
score = game.score()
score and {k: score[k] for k in ('points', 'points_no_time')}


## Play the rest automatically

A bot is one method: given the state, say which action to take.


In [ ]:
from pokelike.bot.base import Bot


class CatchThenFight(Bot):
    """Catch while the team is small, otherwise take the first option."""

    name = 'demo'

    def choose(self, state):
        if len(state.get('team') or []) < 4:
            for i, a in enumerate(state['actions']):
                if a.get('node') == 'catch':
                    return i
        return 0


while not obs.get('done') and obs.get('actions') and game.steps < 200:
    obs = game.step(CatchThenFight().choose(obs))

print(obs['screen'], '| badges', (game.last_alive or {}).get('run', {}).get('badges'))


## Whole runs, and comparing bots

`play` runs one from start to finish and hands back the decision trace. `compare`
runs several bots over the **same** seeds and pairs them up: runs vary enormously
by luck here, so two separate averages mostly measure who drew the nicer maps.


In [ ]:
game.close()   # compare opens its own

from pokelike import compare
from pokelike.bot.random_bot import RandomBot

result = compare({'demo': CatchThenFight(), 'random': RandomBot(seed=0)},
                 seeds=range(5), baseline='random')
print(result['table'])


## Close it

Stops the browser and the server together. Worth actually running: a leaked
browser is what makes the next thing fail, for reasons that have nothing to do
with what you changed.


In [ ]:
game.close()


---

**Next:** [Writing a bot](../../../../README.md#writing-a-bot) for the full
interface, and [the leaderboard](../../../../leaderboard/) if you want to enter
the contest.
